In [23]:
# ============================================================
# Kaggle Notebook: K-Fold Fine-tuning (English → Telugu)
# Model: Helsinki-NLP/opus-mt-en-dra
# Metrics: BLEU + ChrF
# ============================================================

!pip install -q transformers datasets sentencepiece sacrebleu scikit-learn accelerate


In [24]:
import os
import glob
import random
import shutil
import json
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
from sklearn.model_selection import KFold
import datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)


In [25]:
# CONFIGURATION
# =========================
MODEL_NAME = "Helsinki-NLP/opus-mt-en-dra"
K = 5
SEED = 42
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 5e-5
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128
OUTPUT_ROOT = "/kaggle/working/Cross_Validation_2_FineTune_155k"
# =========================

random.seed(SEED)
np.random.seed(SEED)


In [26]:

# -------------------------
# FIND dataset (txt)
# -------------------------
def find_dataset_txt():
    candidates = glob.glob("/kaggle/input/**/*.txt", recursive=True)
    candidates += glob.glob("/kaggle/working/*.txt")
    if not candidates:
        raise FileNotFoundError("❌ No .txt dataset found in /kaggle/input or /kaggle/working.")
    return candidates[0]

dataset_path = find_dataset_txt()
print(f"✅ Using dataset: {dataset_path}")


✅ Using dataset: /kaggle/input/english-telugu-pairs/english_telugu_data.txt


In [27]:
# LOAD DATA
# -------------------------
src_texts, tgt_texts = [], []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("++++$++++")
        if len(parts) != 2:
            continue
        en, te = parts[0].strip(), parts[1].strip()
        if en and te:
            src_texts.append(en)
            tgt_texts.append(te)

print(f"✅ Parsed {len(src_texts)} sentence pairs.")
for i in range(min(3, len(src_texts))):
    print(f"{i+1}) {src_texts[i]}  →  {tgt_texts[i]}")

hf_dataset = datasets.Dataset.from_dict({"en": src_texts, "te": tgt_texts})


✅ Parsed 155798 sentence pairs.
1) His legs are long.  →  అతని కాళ్ళు పొడవుగా ఉన్నాయి.
2) Who taught Tom how to speak French?  →  టామ్ ఫ్రెంచ్ మాట్లాడటం ఎలా నేర్పించారు?
3) I swim in the sea every day.  →  నేను ప్రతి రోజు సముద్రంలో ఈత కొడతాను.


In [28]:
# TOKENIZER & MODEL
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess_examples(examples):
    inputs = tokenizer(examples["en"], max_length=MAX_SOURCE_LENGTH, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["te"], max_length=MAX_TARGET_LENGTH, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs


In [30]:
# Install evaluate if not already
!pip install -q evaluate

import evaluate

# Metrics
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[lbl.strip()] for lbl in labels]  # wrap each reference in a list
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    
    # BLEU
    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)
    # ChrF
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
    
    return {
        "bleu": round(bleu["score"], 4),
        "chrf": round(chrf["score"], 4),
    }


In [32]:
# TRAIN LOOP (per fold)
# -------------------------
indices = list(range(len(hf_dataset)))
fold_summaries = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(indices), start=1):
    print(f"\n====================== Fold {fold_idx}/{K} ======================")
    fold_dir = os.path.join(OUTPUT_ROOT, f"fold_{fold_idx}")
    os.makedirs(fold_dir, exist_ok=True)

    train_ds = hf_dataset.select(train_idx)
    val_ds = hf_dataset.select(val_idx)
    tokenized_train = train_ds.map(preprocess_examples, batched=True, remove_columns=["en", "te"])
    tokenized_val = val_ds.map(preprocess_examples, batched=True, remove_columns=["en", "te"])

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=fold_dir,
        eval_strategy="epoch",
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        predict_with_generate=True,
        save_total_limit=1,
        save_strategy="epoch",
        logging_steps=50,          # smaller logging steps for more frequent updates
        fp16=True,
        seed=SEED,
        load_best_model_at_end=False,
        report_to="none",          # disables wandb/tensorboard if not installed
        disable_tqdm=False,        # force progress bar display
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print("🚀 Training started ...")
    trainer.train()
    print("✅ Training completed.")

    # Save model & tokenizer
    trainer.save_model(fold_dir)
    tokenizer.save_pretrained(fold_dir)

    # Evaluate & save metrics
    print("🔍 Evaluating on validation set ...")
    preds_output = trainer.predict(tokenized_val)
    metrics = preds_output.metrics
    decoded_preds = tokenizer.batch_decode(preds_output.predictions, skip_special_tokens=True)
    preds_file = os.path.join(fold_dir, "val_predictions.txt")
    refs_file = os.path.join(fold_dir, "val_references.txt")
    with open(preds_file, "w", encoding="utf-8") as pf, open(refs_file, "w", encoding="utf-8") as rf:
        for p, r in zip(decoded_preds, val_ds["te"]):
            pf.write(p.strip() + "\n")
            rf.write(r.strip() + "\n")

    # Save metrics
    metrics_path = os.path.join(fold_dir, "eval_metrics.json")
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"✅ Fold {fold_idx} metrics: {metrics}")
    fold_summaries.append({"fold": fold_idx, "metrics": metrics})

    # Reload fresh model for next fold
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# -------------------------
# SAVE SUMMARY + ZIP
# -------------------------
summary_path = os.path.join(OUTPUT_ROOT, "cv_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(fold_summaries, f, indent=2, ensure_ascii=False)

zip_path = "/kaggle/working/Cross_Validation_2_FineTune_155k.zip"
print(f"📦 Zipping everything into {zip_path} ...")
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(base_name=zip_path.replace(".zip", ""), format="zip", root_dir=OUTPUT_ROOT)
print("✅ All done! Download from the Kaggle 'Files' tab.")



====================== Fold 1/5 ======================


Map:   0%|          | 0/124638 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/31160 [00:00<?, ? examples/s]

/tmp/ipykernel_37/1791952963.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


...

Epoch,Training Loss,Validation Loss,Bleu,Chrf
1,0.475900,0.388356,63.566200,82.293700


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3685: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[62951]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


✅ Training completed.
🔍 Evaluating on validation set ...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


✅ Fold 5 metrics: {'test_loss': 0.38835620880126953, 'test_bleu': 63.5662, 'test_chrf': 82.2937, 'test_runtime': 607.9838, 'test_samples_per_second': 51.25, 'test_steps_per_second': 3.204}
📦 Zipping everything into /kaggle/working/Cross_Validation_2_FineTune_155k.zip ...
✅ All done! Download from the Kaggle 'Files' tab.


In [12]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    print(f"✅ Number of GPU devices available: {num_devices}")
    for i in range(num_devices):
        print(f"  - Device {i}: {torch.cuda.get_device_name(i)}")
else:
    num_devices = 1  # CPU
    print("⚠️ No GPU found. Using CPU.")


✅ Number of GPU devices available: 2
  - Device 0: Tesla T4
  - Device 1: Tesla T4


In [33]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# CONFIG
OUTPUT_ROOT = "/kaggle/working/Cross_Validation_2_FineTune_155k"
NUM_FOLDS = 5
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

# Loop over folds
for fold_idx in range(1, NUM_FOLDS + 1):
    fold_dir = f"{OUTPUT_ROOT}/fold_{fold_idx}"
    print(f"\n====================== Fold {fold_idx} ======================")

    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(fold_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(fold_dir)
    model.eval()

    # Loop for multiple sentences
    while True:
        en_sentence = input("Enter English sentence (or 'quit' to stop): ")
        if en_sentence.lower() == "quit":
            break

        # Tokenize input
        inputs = tokenizer(en_sentence, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SOURCE_LENGTH)

        # Generate translation
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=MAX_TARGET_LENGTH, num_beams=4)
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)

        print(f"{en_sentence}  →  {translation}")



====================== Fold 1 ======================


Enter English sentence (or 'quit' to stop):  hello


hello  →  హలో


Enter English sentence (or 'quit' to stop):  how are you


how are you  →  మీరు ఎలా ఉన్నారు?


...

Enter English sentence (or 'quit' to stop):  quit



====================== Fold 4 ======================


Enter English sentence (or 'quit' to stop):  quit



====================== Fold 5 ======================


Enter English sentence (or 'quit' to stop):  quit


In [35]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json
import os

# CONFIG
OUTPUT_ROOT = "/kaggle/working/Cross_Validation_2_FineTune_155k"
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

# -----------------------
# Load CV summary to find best fold
# -----------------------
summary_path = os.path.join(OUTPUT_ROOT, "cv_summary.json")
with open(summary_path, "r", encoding="utf-8") as f:
    fold_summaries = json.load(f)

# Pick best fold based on eval_bleu
best_fold = max(fold_summaries, key=lambda x: x["metrics"].get("eval_bleu", 0))["fold"]
print(f"✅ Best fold selected: Fold {best_fold}")

fold_dir = os.path.join(OUTPUT_ROOT, f"fold_{best_fold}")

# Load model & tokenizer
tokenizer = AutoTokenizer.from_pretrained(fold_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(fold_dir)
model.eval()

# -----------------------
# Interactive testing loop
# -----------------------
while True:
    en_sentence = input("Enter English sentence (or 'quit' to stop): ")
    if en_sentence.lower() == "quit":
        break

    inputs = tokenizer(en_sentence, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SOURCE_LENGTH)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=MAX_TARGET_LENGTH, num_beams=4)

    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"{en_sentence}  →  {translation}")


✅ Best fold selected: Fold 1


Enter English sentence (or 'quit' to stop):  SafetyEye – AI-Powered Workplace Occupancy & Safety Monitor


SafetyEye – AI-Powered Workplace Occupancy & Safety Monitor  →  భద్రతా సేఫ్ - ఐఎ-అనర్ సిబ్బంది ఆక్యునేషనల్ మరియు భద్రతా పర్యవేక్షకుడు


Enter English sentence (or 'quit' to stop):  This tool helps office and industrial space managers improve space utilization while ensuring employees follow safety protocols.


This tool helps office and industrial space managers improve space utilization while ensuring employees follow safety protocols.  →  ఈ టూత్‌పేస్ట్ కార్యాలయానికి మరియు పరిశ్రమ నిర్వాహకులకు సహాయపడుతుంది, ఆ సమయంలో ఉద్యోగులు భద్రతా ప్రాక్టీస్‌ను అనుసరిస్తున్నారు.


Enter English sentence (or 'quit' to stop):  quit


In [39]:
import shutil
import os

os.makedirs("/kaggle/working/outputs", exist_ok=True)
shutil.copy("/kaggle/working/Cross_Validation_2_FineTune_155k.zip", "/kaggle/working/outputs/")


'/kaggle/working/outputs/Cross_Validation_2_FineTune_155k.zip'

In [51]:
import shutil
import os

zip_src = "/kaggle/working/Cross_Validation_2_FineTune_155k.zip"
zip_dst = "/kaggle/outputs/Cross_Validation_2_FineTune_155k.zip"

# Make outputs folder if it doesn't exist
os.makedirs("/kaggle/outputs/", exist_ok=True)

# Move the ZIP to outputs
shutil.move(zip_src, zip_dst)
print("✅ File moved to /kaggle/outputs/")


✅ File moved to /kaggle/outputs/


In [58]:
os.path.exists("/kaggle/outputs/Cross_Validation_2_FineTune_155k.zip")


True

In [70]:
import dropbox
from dropbox.files import WriteMode

ACCESS_TOKEN = "sl.u.AGB3IqwEhj2P0wQV95kSR6WGXXormhyzTNLldbZQjJo10QoIrqXaX-Y94iQFPZjPUv4BS4HgfOnKBY3GfNlvkMoE6XcHMFB5V2OWIXsVy2cGKp8K_XbuL9UxybSl7PwJQhL3DiXk-vcg99FhEUmtltPEOnDOCOCIALIAdxBmwBF_x4xX8WUu5Rwa26EeRVRvnm3MlPaWwKEAzavyHFbt7bxlVvx4XoWs78-F__epTmf7upBTCqYyOZwF2lxeYOsg3rw7rW6fP0bPPvMA4j9Oyf-GLimCdr0hR-Nuvn4emWW_-8C5G15BS9QDABDTRDXx0oPuFA6grsX67FpEmJIfnsCr5RFALcDdPG0R2PcwhtoxtguzI9xcTg-ieJWuGLgbQqcHMI0A6hO3X7UsaKAlzgrYVq2yU2xNnJGCKP3RFwWepfnj0TtqqWXALD7otn0pAirLf7Dns5Y_vGOaLVmX60oqakjM8h4sRyEROUjlGO_kKuwsEX2ZZv8Kg6C9WPcC5AJeXg-BIuMJhK0spjP6Z_ZBXzjNNvIqn-QqL1wkkoqjwCUJX_CM-NoMeqsfJYikgO-iFJeOKcCa0Kd46mFE3tpWq9YH3XUgJMUlISnubdCgtV4R8HUK53Thme2hTT73wPN8UB-0AehEV71OT2RuShLsV43wO1IFNXO_QaJIOu8KjlfR13Ih8hOiaue52LqE0FDwCheAxM3vGMKo0Gkr4QiETKJzpL79NI6a_nPji0kCnI3exm5CncD0U70ox_Pnhw4Nlka6TYe9U0v2AnjIkRLporUqqXtSIZAqLV5leOJoIX059lcNDWyQS15R1yd02jDxFtiP6Pw18o-Ov76Bs5jiw7jmeh0ziErOq4P9RObfj3Tx7HxzjUI_y5euS-VStuXyF4knr4HaGGr01gJa-EciSFnSeR1xDHwsUmLKpiXlqUL7m8JQpxR0Nnqf0SJFt2Y9KTe-n4PHW2M_zsVCO__thASUbCba3U9ofGXaEVxTMO4nHyvUnmhUi17Uf62WM952iw0JTJLgrE1TgybKkqnV3IwN5nwn8Nf7giU69LuQZA_le5gdgw5Zl0z4udBJg7LWd0UIUE4B0rdHe3YbZzUw6EeagiRfqX4ISw4WIvHNUJExVjeMpaL42JVEYzQAXSfnA6gm617xC_tb-zm-sfAM_S5hrpBKXoWQkcHhA19RBHXnbVdLO6tWGU_-0-b_W3VB1KzzhkMoY8jArVMT36XLSUJICPytsNY66UgMqMmn8t3KwKM53hxJHTNwHAlMlen6mJ88VO-P7liYF0t49ARXzkShx24_ZparjskjwcaTazTEa8RdMIVnxwEzHWtcJTVWlvDC9-LGqmufJ3HA1dHDyW0isbVtwpfHxN9YVLgQ5-vkfWsdOejUbsElYLKiULYfBkh0iBCDUXnxGtrj2F3zClUhtM5OcE8eXvbUtdmlfA"
dbx = dropbox.Dropbox(ACCESS_TOKEN)

local_path = "/kaggle/outputs/Cross_Validation_2_FineTune_155k.zip"
dropbox_path = "/Cross_Validation_2_FineTune_155k.zip"

CHUNK_SIZE = 150 * 1024 * 1024  # 150 MB per chunk

with open(local_path, "rb") as f:
    file_size = f.seek(0, 2)  # get size
    f.seek(0)
    
    if file_size <= CHUNK_SIZE:
        dbx.files_upload(f.read(), dropbox_path, mode=WriteMode('overwrite'))
    else:
        upload_session_start_result = dbx.files_upload_session_start(f.read(CHUNK_SIZE))
        cursor = dropbox.files.UploadSessionCursor(session_id=upload_session_start_result.session_id,
                                                   offset=f.tell())
        commit = dropbox.files.CommitInfo(path=dropbox_path, mode=WriteMode('overwrite'))

        while f.tell() < file_size:
            if (file_size - f.tell()) <= CHUNK_SIZE:
                dbx.files_upload_session_finish(f.read(CHUNK_SIZE), cursor, commit)
            else:
                dbx.files_upload_session_append_v2(f.read(CHUNK_SIZE), cursor)
                cursor.offset = f.tell()

print("✅ Large file uploaded to Dropbox!")


✅ Large file uploaded to Dropbox!


In [ ]:
/kaggle/outputs/Cross_Validation_2_FineTune_155k.zip